## Step 5

Inputs: s3://thesis--ec331-s3/merged-price-volume-bids/
Output: s3://thesis--ec331-s3/capped-volume-bids/  

What does this do?  
This script corrects for the phenenomenon where some firms bid more than their max availability, and then rely on the dispatch algorithm to account for this, which means we can't rely

In [1]:
import pandas as pd
import numpy as np
import awswrangler as wr
import time

# Add timing and logging
print("Starting data load...")
start_time = time.time()

# Read Parquet files from the specific S3 folder
s3_input_path = "s3://thesis--ec331-s3/enriched-volume-bids/RAISE1SEC_PUBLIC_DVD_BIDPEROFFER2_202310010000_enriched/"
s3_output_path = "s3://thesis--ec331-s3/step-2-capped-volume-bids/"

# Load the data
df = wr.s3.read_parquet(path=s3_input_path)
print(f"Data loaded in {time.time() - start_time:.2f} seconds")
print(f"DataFrame shape: {df.shape}")
print(f"Memory usage: {df.memory_usage().sum() / 1024 / 1024:.2f} MB")

# Print some sample data to understand structure
print("\nSample data:")
sample = df.sample(3)
print(sample[["DUID", "TRADINGDATE", "PERIODID", "MAXAVAIL"] + 
      [f"PRICEBAND{i}" for i in range(1, 11)] + 
      [f"BANDAVAIL{i}" for i in range(1, 11)]])

# Define a variable to count capped rows
total_capped_rows = 0

# Define a truly vectorized approach for capping
def cap_bands_vectorized(df_group):
    """Process a group of bids with the same DUID, TRADINGDATE, PERIODID"""
    global total_capped_rows
    
    # Create a copy to avoid modifying the original
    result_df = df_group.copy()
    
    # For each row in the group
    for idx, row in result_df.iterrows():
        max_avail = row["MAXAVAIL"]
        
        # Extract prices and volumes
        prices = [row[f"PRICEBAND{i}"] if f"PRICEBAND{i}" in row and not pd.isna(row[f"PRICEBAND{i}"]) else np.nan 
                 for i in range(1, 11)]
        volumes = [row[f"BANDAVAIL{i}"] if f"BANDAVAIL{i}" in row and not pd.isna(row[f"BANDAVAIL{i}"]) else 0.0 
                  for i in range(1, 11)]
        
        # Create a list of (price, volume, band_index)
        bands = [(p, v, i) for i, (p, v) in enumerate(zip(prices, volumes), 1) if not pd.isna(p)]
        
        # Sort by price
        bands.sort(key=lambda x: x[0])
        
        # Apply capping
        running_sum = 0.0
        capped_volumes = [0.0] * 10
        row_was_capped = False
        
        for price, volume, band_idx in bands:
            if running_sum >= max_avail:
                # If we're here and there's still volume to allocate, we need to cap
                if volume > 0:
                    row_was_capped = True
                break
                
            remaining = max_avail - running_sum
            if volume > remaining:
                row_was_capped = True
                
            capped_volume = min(volume, remaining)
            
            capped_volumes[band_idx-1] = capped_volume
            running_sum += capped_volume
        
        # Count this row if it was capped
        if row_was_capped:
            total_capped_rows += 1
        
        # Update the result dataframe
        for i in range(1, 11):
            col = f"BANDAVAIL{i}"
            if col in result_df.columns:
                result_df.at[idx, col] = capped_volumes[i-1]
    
    return result_df

# Process data in chunks
def process_data(df, chunk_size=1000):
    print("\nProcessing data in chunks...")
    capped_dfs = []
    total_chunks = (len(df) + chunk_size - 1) // chunk_size
    
    for i in range(0, len(df), chunk_size):
        chunk_start = time.time()
        print(f"Processing chunk {i//chunk_size + 1}/{total_chunks}")
        
        # Get chunk
        chunk = df.iloc[i:i+chunk_size].copy()
        
        # Process chunk - group by key columns
        grouped = chunk.groupby(["DUID", "TRADINGDATE", "PERIODID"])
        capped_chunk = grouped.apply(cap_bands_vectorized)
        
        capped_dfs.append(capped_chunk)
        print(f"Chunk {i//chunk_size + 1} completed in {time.time() - chunk_start:.2f} seconds")
    
    # Combine results
    if capped_dfs:
        return pd.concat(capped_dfs)
    else:
        return pd.DataFrame()

# Execute the processing
capped_df = process_data(df, chunk_size=500)

# Print capping statistics
print(f"\nTotal rows processed: {len(df)}")
print(f"Total rows capped: {total_capped_rows} ({total_capped_rows/len(df)*100:.2f}%)")
print(f"\nAll processing completed in {time.time() - start_time:.2f} seconds")
print(capped_df.head())

# Extract the date part from the input path for the output path
import re
date_match = re.search(r'(\d{12})', s3_input_path)
date_str = date_match.group(1) if date_match else "processed"

# Create output path with the same naming structure
output_folder = f"{s3_output_path}RAISE1SEC_PUBLIC_DVD_BIDPEROFFER2_{date_str}_capped/"

# Write the results to the output S3 bucket
print(f"\nWriting results to {output_folder}")
wr.s3.to_parquet(
    df=capped_df,
    path=output_folder,
    dataset=True,
    mode='overwrite',
    compression='snappy'
)

print(f"Process completed in {time.time() - start_time:.2f} seconds")
print(f"Results written to {output_folder}")
print(f"Summary: {total_capped_rows} out of {len(df)} rows were capped ({total_capped_rows/len(df)*100:.2f}%)")

Starting data load...
Data loaded in 87.41 seconds
DataFrame shape: (21227616, 47)
Memory usage: 7611.83 MB

Sample data:


KeyError: "['PRICEBAND1', 'PRICEBAND2', 'PRICEBAND3', 'PRICEBAND4', 'PRICEBAND5', 'PRICEBAND6', 'PRICEBAND7', 'PRICEBAND8', 'PRICEBAND9', 'PRICEBAND10'] not in index"